In [1]:
import pyGinkgo as pg
import numpy as np

### Direct solver bindings

In [ ]:
fn = 'm1.mtx'
dev = pg.device("CUDA")
mtx = pg.read(path=fn, dtype="double", format="Csr")
n_rows = mtx.size[0]

b = pg.as_tensor(
  device=dev, dim=(n_rows,1), dtype="double", fill=1.0
)
x = pg.as_tensor(
  device=dev, dim=(n_rows,1), dtype="double", fill=0.0
)

# Create ILU preconditioner
preconditioner = pg.preconditioner.Ilu(dev, mtx)

# Setup GMRES solver
solver = pg.solver.gmres(dev, mtx, preconditioner=preconditioner,
    max_iters=1000, krylov_dim=30, reduction_factor=1e-06
)

#Apply
logger, result = solver.apply(b, x)

In [3]:
result_cpu = result.copy_to_host()

/home/ubuntu/projects/work/cpp/pyGinkgo/src/cpp_bindings/dense.cppWarning creating a copy of dense


In [5]:
for i in range(10):
    print(result_cpu.at(i, 0), end=", ")

0.6044527378681241, 0.8250277658143181, 0.9154476498996701, 0.9545267011464476, 0.9719330030003778, 0.9798380401275002, 0.9834770837870621, 0.9851691223654254, 0.9859619246225143, 0.9863356545781781, 

### Config solver

In [ ]:
args = {
    "type": "solver::Gmres",
    "krylov_dim": 30,
    "preconditioner": {
        "type": "preconditioner::Jacobi",
        "max_block_size": 1
    },
    "criteria": [
        {"type": "Iteration", "max_iters": 1000},
        {
          "type": "ResidualNorm", 
          "reduction_factor": 1e-6, 
          "baseline": "rhs_norm"
        }
    ],
}